# ATLAS Diphoton Hackathon — Boilerplate

**Goal.** Build the best possible selection to reveal the **signal** in the *blackbox* dataset — a small resonant peak sitting in the diphoton invariant mass $m_{\gamma\gamma}$ — **without sculpting a fake bump** in $m_{\gamma\gamma}$.

**Submission.** A CSV listing the `event_id`s you **keep** (one per line, header `event_id`).

**Scoring.** A closed evaluator fits a smooth background (log-quadratic) to the $m_{\gamma\gamma}$ sidebands of your selection and counts the excess in the signal region [120, 130] GeV.
- `Z_data` = significance of the excess over the sideband fit — **your leaderboard score** (higher is better).
- `Z_bg`   = the same fit run on the *background-only* part of your selection — must stay **below 3σ**, otherwise your selection sculpted the background and the submission is rejected.

Anything goes — simple cuts, a classifier trained between the signal and background MC, a classifier trained between signal MC and the data sidebands, an anomaly score, ... The trick is to enhance the signal **without** carving a bump into the smooth background.

## Datasets

Four parquet files. Every event has a synthetic `event_id` (0..N−1 within each file).

| File | Description |
|---|---|
| `blackbox.parquet`    | The events you must classify. A mix of signal and background. **No labels.** Submit the `event_id`s you keep. |
| `pseudodata.parquet`  | A second, independent unlabelled mix with the same composition — explore freely. |
| `signal.root`      | **Signal** Monte Carlo. |
| `background.root`  | **Background** Monte Carlo. |

All files share the same feature schema. **Units:** momenta/masses (`*_pt`, `*_m`, `diphoton_mass`, `met_pt`, `HT_30`) are in **MeV** — divide by 1000 for GeV.

**Pre-applied cuts** (already in every file): ≥2 photons; $105 < m_{\gamma\gamma} < 160$ GeV.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "/pscratch/sd/d/dnoll/projects/ml4fp/atlas"

def myy_gev(df):
    """Diphoton invariant mass in GeV (stored in MeV)."""
    return df["diphoton_mass"].to_numpy() / 1000.0

In [ ]:
blackbox      = pd.read_parquet(f"{DATA_DIR}/blackbox.parquet")
pseudodata    = pd.read_parquet(f"{DATA_DIR}/pseudodata.parquet")

print(f"blackbox      : {len(blackbox):,} events, {blackbox.shape[1]} cols")
print(f"pseudodata    : {len(pseudodata):,} events")
blackbox.head()

In [ ]:
# Every available feature:
for i, c in enumerate(blackbox.columns):
    print(f"{i:3d}  {c}")

## Quick look at $m_{\gamma\gamma}$

The blackbox is a tiny signal peak on a large, smoothly falling background. Signal MC peaks in the signal region; background MC is smooth.

In [ ]:
edges = np.linspace(105, 160, 56)
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(myy_gev(blackbox),      bins=edges, histtype="step", density=True, label="blackbox")
ax.axvspan(120, 130, color="orange", alpha=0.15, label="signal region")
ax.set_xlabel(r"$m_{\gamma\gamma}$ [GeV]")
ax.set_ylabel("normalized / bin")
ax.legend()
fig.tight_layout()

## Your turn

Write a `select(df) -> boolean mask` that flags the events you keep in `blackbox`.

- **Maximize** `Z_data` (the signal significance).
- **Keep** `Z_bg < 3` (don't sculpt the $m_{\gamma\gamma}$ shape).

Use any technique: kinematic cuts, BDTs/MLPs trained on signal vs. background MC, classifiers trained signal-MC vs. data-sidebands, anomaly scores, ...

**Watch out:** anything correlated with $m_{\gamma\gamma}$ — e.g. cutting near the peak — sculpts a fake bump into the background and gets rejected. Mass decorrelation is half the challenge.

In [ ]:
def select(df: pd.DataFrame) -> np.ndarray:
    """Return a boolean mask of the events to KEEP. Replace this baseline."""
    # Baseline: keep everything (a valid but unoptimised submission).
    return np.ones(len(df), dtype=bool)

mask = select(blackbox)
print(f"keeping {mask.sum():,} / {len(blackbox):,} events ({100*mask.mean():.1f}%)")

## Submission

Write the `event_id`s you keep to a CSV and submit it.

In [ ]:
def write_submission(df, mask, path="submission.csv"):
    mask = np.asarray(mask).astype(bool)
    if len(mask) != len(df):
        raise ValueError(f"mask has {len(mask)} rows, expected {len(df)}")
    kept = df.loc[mask, "event_id"].astype(np.int64)
    kept.to_csv(path, index=False)
    print(f"wrote {path}: {len(kept):,} kept event_ids")

write_submission(blackbox, mask, "submission.csv")